In [1]:
import re
import ast
import pandas as pd

In [2]:
!ls

gfmag_collect_urls.py  ickg_llm.py	  local_llm.py	__pycache__
gfmag_get_text.py      llm_extraction.py  parser.py	testing.ipynb


In [3]:
text = '''=== ARTICLE 118 ===
DATE: 2020-09-11T00:00:00+00:00
URL: https://gfmag.com/economics-policy-regulation/quintessential-capital-management-gabriel-grego/
TRIPLETS_RAW:
[['Quintessential Capital Management:financial_institution', 'has_positive_impact','short activism:event', 'Finance'], ['Gabriel Grego:person', 'is_member_of', 'Quintessential Capital Management:financial_institution', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to', 'company fraud:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','management team breaking the law:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','serious crimes:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','suspicion of misbehavior:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','regulators and legal authorities:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to', 'bad ap'''

In [4]:
with open("../data/sustainable_kg_triplets_1000_raw.txt", "r", encoding="latin-1") as f:
    text = f.read(10000)

In [5]:
    #articles = re.split(r"(?=== ARTICLE \d+ ===)", text)[1:]
    
    #for art in articles: 
    date = re.search(r"DATE:\s*(.+)", text)
    date = date.group(1).strip() if date else None
    
    url = re.search(r"URL:\s*(.+?)(?=TRIPLETS_RAW|\n|$)", text, flags=re.S)
    url = url.group(1).strip() if url else None
    
    # including possible truncation
    tr = re.search(r"TRIPLETS_RAW:\s*(\[.*)", text, flags=re.S)
    raw_triplets = tr.group(1) if tr else ""
    
    # ensure it starts with '['
    if not raw_triplets.startswith("["):
        raw_triplets = "[" + raw_triplets
        
    # cut off incomplete tail at last ']'
    last_bracket = raw_triplets.rfind("]")
    raw_triplets = raw_triplets[: last_bracket + 1] + "]"

In [8]:
text_df = parse_text(path="../data/sustainable_kg_triplets_1000_raw.txt")

In [9]:
text_df["TRIPLETS"][0]

"[['Renewable energy:economic_indicator', 'is_competitor_of', 'Coal:economic_indicator', 'Energy'], ['Solar:product', 'has_positive_impact', 'Renewable energy:economic_indicator', 'Energy'], ['Wind:product', 'has_positive_impact', 'Renewable energy:economic_indicator', 'Energy'], ['Solar:product', 'has_positive_impact', 'Electricity demand:economic_indicator', 'Energy'], ['Renewable energy:economic_indicator', 'is_competitor_of', 'Coal:economic_indicator', 'Energy'], ['China:country', 'is_member_of', 'Renewable energy:economic_indicator', 'Energy'], ['India:country', 'is_member_of', 'Renewable energy:economic_indicator', 'Energy']]"

In [10]:
raw = "[['Renewable energy:economic_indicator', 'is_competitor_of', 'Coal:economic_indicator', 'Energy'], ['Solar:product', 'has_positive_impact', 'Renewable energy:economic_indicator', 'Energy'], ['Wind:product', 'has_positive_impact', 'Renewable energy:economic_indicator', 'Energy'], ['Solar:product', 'has_positive_impact', 'Electricity demand:economic_indicator', 'Energy'], ['Renewable energy:economic_indicator', 'is_competitor_of', 'Coal:economic_indicator', 'Energy'], ['China:country', 'is_member_of', 'Renewable energy:economic_indicator', 'Energy'], ['India:country', 'is_member_of', 'Renewable energy:economic_indicator', 'Energy']]"
triplets = ast.literal_eval(raw)

# functions

In [ ]:
# TODO: category, object types

In [7]:
def parse_text(path:str):
    """ Parse ARTICLE blocks into clean dictionaries and write them into a dataframe"""
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        text = f.read()
        
    articles = re.split(r"(?=== ARTICLE \d+ ===)", text)[1:]
    
    for art in articles: 
        date = re.search(r"DATE:\s*(.+)", art)
        date = date.group(1).strip() if date else None
        
        url = re.search(r"URL:\s*(.+?)(?=TRIPLETS_RAW|\n|$)", art, flags=re.S)
        url = url.group(1).strip() if url else None
        
        # including possible truncation
        tr = re.search(r"TRIPLETS_RAW:\s*(\[.*)", art, flags=re.S)
        raw_triplets = tr.group(1) if tr else ""
    
        # ensure it starts with '['
        if not raw_triplets.startswith("["):
            raw_triplets = "[" + raw_triplets
            
        # cut off incomplete tail at last ']'
        if raw_triplets:
            last_bracket = raw_triplets.rfind("]") 
            if last_bracket != -1:
                # Find FIRST of the last brackets (second-to-last ])
                second_last = raw_triplets.rfind("]", 0, last_bracket)
                cut_pos = second_last + 1 if second_last != -1 else 0
                
                raw_triplets = raw_triplets[:cut_pos] + "]"
            else:
                # No brackets at all, just add one
                raw_triplets += "]"
        #triplets = ast.literal_eval(raw_triplets)
    
        rows.append({"DATE": date, "URL": url, "TRIPLETS": raw_triplets})

    return pd.DataFrame(rows)


In [11]:
ca_test = text_df["TRIPLETS"][54]

In [12]:
text_df = parse_text(path="../data/sustainable_kg_triplets_1000_raw.txt")

In [13]:
def fix_syntax(raw):
    """'A':'B' → 'A:B' for valid ast.literal_eval"""
    return re.sub(r"'([^']*)'\s*:\s*'", r"'\1:", raw, flags=re.DOTALL)

fixed = fix_syntax(raw)
triplets = ast.literal_eval(fixed)

In [16]:
text_df["TRIPLETS"][54]

"[['Central America':'region', 'has_negative_impact', 'dry corridor': 'economic_indicator', 'Central America'], ['Central America':'region', 'has_negative_impact', 'drought': 'economic_indicator', 'Central America'], ['Panama': 'country', 'has_negative_impact', 'drought': 'economic_indicator', 'Panama'], ['Panama': 'country', 'has_negative_impact', 'canal': 'economic_indicator', 'Panama'], ['Panama': 'country', 'has_negative_impact', 'petroleum costs': 'economic_indicator', 'Panama'], ['Costa Rica': 'country', 'is_member_of', 'Banco': 'financial_institution', 'Costa Rica']]"

In [15]:
df = text_df